# Run 2 (1 year): true sky envelope, wavelet RJMCMC vs the OG chunked pipeline on the same data

Same test as the 30-day [run 2](../../30_days/run2/run2_sky_vs_og.ipynb), over **one year**. Inject the Galactic
foreground modulated by the true sky envelope $P_c(t)=M_c^2(t)$ (`lat, long, psi, s1, s2` of
[config/sources.yaml](../../30_days/config/sources.yaml), paper-2 values with $\sin\psi=-0.83$) into **one** 1-year time series,
and analyse that series with both pipelines:

* **wavelet:** WDM transform ($N_f=3600$, 876 pixels of 10 h), Gamma likelihood, free wavelet atoms per channel
  (reversible jump), eryn, settings in [config/wavelet.yaml](../config/wavelet.yaml);
* **OG:** 24 chunks of 15.21 d cut from the same series (365 d / 24; the chunks must tile the series, so not 15 d),
  5% Tukey taper, log-binned periodograms, Gamma likelihood, 5 sky parameters, nessai, settings in
  [config/og_config.yaml](../config/og_config.yaml).

The OG pipeline is run twice: with the **correct Gamma likelihood** (`outputs/run2/og/`) and with the **paper's**
(Pozzoli et al. Eq. 9, `--legacy-dof`, `outputs/run2/og_legacy/`). The compared quantity is the galactic power relative
to the true amplitude, $10^{\log_{10}A-\log_{10}A_{\rm true}}\,P_c(t)$, whose truth is $M_c^2(t)$.

Needs the 1-year reference fit from [run 1](../run1/run1_inject_recover.ipynb) (for the wavelet amplitude reference).
Set `RERUN = True` to simulate and sample again: both samplers see about 12 times more data than over 30 days. Run the
OG pipeline from a terminal (forked nessai workers deadlock under JAX), from the repository root:

    python simulations/common/og.py --config simulations/1_year/config/wavelet.yaml --tag run2
    python simulations/common/og.py --config simulations/1_year/config/wavelet.yaml --tag run2 --legacy-dof

In [ ]:
import sys
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt
import corner
import yaml

ROOT = Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
HERE = ROOT / 'simulations' / '1_year'
sys.path.insert(0, str(ROOT / 'simulations' / 'common'))
import injection as I
import og as OG
import diagnostics as D
from bahamas.wavelet.model import DAY
from bahamas.wavelet.sampler import modulation_draws
from bahamas.wavelet.simulate import power_modulation, ENVELOPE_PARAMS

RERUN = False
CFG = I.load_config(HERE / 'config' / 'wavelet.yaml')
if not Path(I.reference_path(CFG)).exists():
    raise FileNotFoundError('no 1-year reference fit: run run1/run1_inject_recover.ipynb first (section 0)')
TAG = 'run2'
OUT = HERE / 'outputs' / TAG
CH = ['A', 'E']
C_TRUTH, C_WAV, C_WAV_L, C_OG, C_OG_L, C_MUTED = '#0b0b0b', '#2a78d6', '#86b6ef', '#eb6834', '#f5b48f', '#52514e'
RUN_K_TRACE = False      # rerun with the burn-in stored (section 2), as long again as the wavelet run
T_CHUNK = yaml.safe_load(open(CFG['og_config']))['chunk']['duration']
CHUNK_EDGES = np.arange(1, round(yaml.safe_load(open(CFG['og_config']))['T'] / T_CHUNK)) * T_CHUNK / DAY
def chunk_lines(ax):
    for e in CHUNK_EDGES:
        ax.axvline(e, color=C_MUTED, ls=':', lw=0.6)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False,
                     'axes.spines.right': False, 'lines.linewidth': 1.5, 'legend.frameon': False})
cfg = CFG
amp_true, amp_ref = I.true_amp(), I.load_reference(I.reference_path(CFG))['amp']
truth_src = {p['name']: p['injected'] for s in I.og_sources().values() for p in s}
print(f'true amp {amp_true:.3f}; wavelet posterior reference amp {amp_ref:.3f}')
print('sky parameters:', I.sky_params())

## 1. Data

In [ ]:
data, inj = I.load_or_simulate('sky', TAG, CFG, rerun=RERUN)
like, post = I.build_posterior(data, CFG)
t_lo, t_hi = float(like.times[0]), float(like.times[-1])     # centres of the first/last analysed time groups
eb, dT = cfg['wavelet']['edge_bins'], cfg['wavelet']['Nf'] * cfg['dt']
print(f"series: {data.Nt * data.Nf * data.dt / DAY:.2f} d; WDM {data.Nt} x {data.Nf} pixels (dT = {dT / 3600:.0f} h)")
print(f"wavelet likelihood: edge_bins = {eb} (drops {eb * dT / 3600:.0f} h at each end), {len(like.times)} time groups, "
      f"centres {t_lo / DAY:.2f}-{t_hi / DAY:.2f} d; {like.ngroups} frequency groups")
t_fine = np.linspace(0, data.Nt * dT, 2000)
P_true = np.stack([np.asarray(power_modulation(I.sky_params(), t_fine, c)) for c in (0, 1)])

## 2. Wavelet run: sampling and convergence

The autocorrelation times are estimated twice (walker-averaged ACF with Sokal's window, and batch means) and the larger
is kept; the run is usable when it holds more than 50 $\tau$ for every quantity ([diagnostics.py](../../common/diagnostics.py)).

In [ ]:
res_path = OUT / 'chains.npz'
if RERUN or not res_path.exists():
    I.run('sky', TAG, CFG)
r = dict(np.load(res_path, allow_pickle=True))
nsteps, nw = int(r['nsteps']), int(r['nwalkers'])
print(f"runtime {float(r['runtime_s']) / 60:.1f} min, {nsteps} steps x {nw} walkers (cold chain); acceptance "
      f"{np.mean(r['acceptance_fraction']):.3f}, RJ acceptance {np.mean(r['rj_acceptance_fraction']):.3f}\n")
diag, stuck = D.report(r, post, np.array([60., 180., 300.]) * DAY)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for c, col in zip(CH, [C_WAV, C_OG]):
    axes[0].plot(r[f'mod_{c}_nleaves'].reshape(nsteps, nw).mean(1), color=col, lw=0.6, label=f'mean K, {c}')
axes[0].set_xlabel('stored step'); axes[0].set_ylabel('mean K over walkers'); axes[0].legend(fontsize=8)
ll = r['log_like'].reshape(nsteps, nw)
axes[1].plot(np.median(ll, 1) - np.median(ll), color=C_WAV, lw=0.6)
axes[1].set_xlabel('stored step'); axes[1].set_ylabel(r'median $\ln L$ − overall')
w = 0.38
print()
for i, (c, col) in enumerate(zip(CH, [C_WAV, C_OG])):
    K, p = r[f'mod_{c}_bayes_K'], r[f'mod_{c}_bayes_posterior']
    axes[2].bar(K + (i - 0.5) * w, p, width=w, color=col, label=c)
    print(c, ' '.join(f'p(K={k})={q:.3f}' for k, q in zip(K, p) if q > 1e-3))
axes[2].set_xlabel('K'); axes[2].set_ylabel('posterior'); axes[2].set_xlim(-0.6, post.max_wavelets + 0.6); axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'convergence_K.png', dpi=150)

### Exploration of K, including the burn-in

eryn runs **32 walkers in each of 4 temperatures (128 in total)**; only the 32 cold walkers ($\beta = 1$) sample the
posterior. `chains.npz` has no burn-in (eryn discards it), so the sampler was rerun with the same seeds and the
burn-in stored (`RUN_K_TRACE = True`, `outputs/run2/k_trace.npz`); the flag
`matches_chains` checks that the stored part of the rerun is the same chain as `chains.npz`. The true envelope is not a
finite sum of atoms, so there is no true K here; the dotted line marks the posterior mode. Per step: the band from the
minimum to the maximum K over the 32 cold walkers, the 25–75% range and the mean; right: the first 500 steps.

In [ ]:
C_BAND_K, C_POST_K = C_WAV_L, C_WAV
kt_path = Path(I.k_trace_path(TAG, CFG))
if RUN_K_TRACE and not kt_path.exists():
    I.run_k_trace('sky', TAG, config=CFG)
if not kt_path.exists():
    print('no k_trace.npz yet: set RUN_K_TRACE = True to make it')
else:
    kt = dict(np.load(kt_path))
    burn, nw_t, ntemps = int(kt['burnin']), int(kt['nwalkers']), int(kt['ntemps'])
    print(f"{nw_t} walkers x {ntemps} temperatures = {nw_t * ntemps} walkers; posterior from the {nw_t} cold walkers")
    print(f"{burn} burn-in + {int(kt['nsteps'])} stored steps; rerun reproduces chains.npz: {bool(kt.get('matches_chains'))}")

    fig, axes = plt.subplots(2, 2, figsize=(11, 5.6), sharey='row', gridspec_kw={'width_ratios': [2.2, 1]})
    for row, c in enumerate(CH):
        K = kt[f'K_{c}'][:, 0].astype(float)              # cold chain: (steps, walkers)
        step = np.arange(1, len(K) + 1)
        lo, hi = K.min(1), K.max(1)
        q25, q75 = np.percentile(K, [25, 75], axis=1)
        mode = int(r[f'mod_{c}_bayes_K'][np.argmax(r[f'mod_{c}_bayes_posterior'])])
        left0 = np.argmax(lo >= 1) + 1                    # first step every walker has at least one atom
        for ax, sl, xmax in zip(axes[row], [slice(None), slice(0, 500)], [len(K), 500]):
            ax.fill_between(step[sl], lo[sl], hi[sl], step='mid', color=C_BAND_K, alpha=0.45, lw=0, label='min–max over cold walkers')
            ax.fill_between(step[sl], q25[sl], q75[sl], step='mid', color=C_POST_K, alpha=0.35, lw=0, label='25–75%')
            ax.plot(step[sl], K[sl].mean(1), color=C_POST_K, lw=1.2, label='mean')
            ax.axhline(mode, color=C_OG, ls=':', lw=1.2, label='posterior mode of K')
            ax.axvspan(0, min(burn, xmax), color=C_MUTED, alpha=0.10, lw=0)
            ax.set_xlim(0, xmax)
        axes[row, 0].text(burn / 2, 0.95, 'burn-in', transform=axes[row, 0].get_xaxis_transform(), ha='center',
                          va='top', fontsize=8, color=C_MUTED)
        axes[row, 0].set_ylabel(f'K, channel {c}')
        print(f'{c}: K range over cold walkers, burn-in {int(lo[:burn].min())}-{int(hi[:burn].max())}, '
              f'stored {int(lo[burn:].min())}-{int(hi[burn:].max())}; every walker off K = 0 from step {left0}; '
              f'mean K: last 500 burn-in steps {K[burn - 500:burn].mean():.2f}, stored {K[burn:].mean():.2f}')
    for ax in axes[1]:
        ax.set_xlabel('step (burn-in included)')
    fig.legend(*axes[0, 0].get_legend_handles_labels(), fontsize=8, ncol=4, loc='lower center', bbox_to_anchor=(0.5, -0.04))
    axes[0, 1].set_title('first 500 steps', fontsize=9)
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(OUT / 'K_trace_burnin.png', dpi=150, bbox_inches='tight')

## 3. OG run on the same series

nessai with the settings of [config/og_config.yaml](../config/og_config.yaml); output in `outputs/run2/og/` (correct dof) and `outputs/run2/og_legacy/` (paper dof). The OG
posterior-predictive galactic power is $10^{\log_{10}A-\log_{10}A_{\rm true}}\,M_c^2(t;\,{\rm lat, long, psi, s1, s2})$.

In [ ]:
# Degrees of freedom of the binned periodogram, on the run-2 OG data: count * var(binned / model) is 1 if each
# |d_k|^2 has 2 dof (exponential), 2 if it has 1 (what Pozzoli et al. Eq. 9 assumes)
sp = OG.chunk_spectra(*I.load_timeseries(TAG, CFG), OG.og_config(TAG, config=CFG))
v = np.concatenate([(b['count'] * (b['ratio'] - 1) ** 2)[:, b['f'] > 1e-3].ravel() for b in OG.binned_vs_model(sp)])
print(f'count * (binned/model - 1)^2 above 1 mHz: {v.mean():.3f} +- {v.std() / np.sqrt(len(v)):.3f}  (2 dof -> 1, 1 dof -> 2)\n')

OG_RUNS = {1: 'OG (paper dof)', 2: 'OG (2 dof)'}     # 1: og_legacy/, 2: og/ (the bahamas default)
ogs = {}
for fac, label in OG_RUNS.items():
    og_path = OUT / OG.og_name(legacy=(fac == 1)) / 'result.hdf5'
    if RERUN or not og_path.exists():
        OG.run(TAG, legacy=(fac == 1), config=CFG)
    with h5py.File(og_path, 'r') as f:
        ogs[fac] = f['posterior_samples'][()]
        print(f"{label}: {len(ogs[fac])} posterior samples, log Z = {f['log_evidence'][()]:.2f} +- "
              f"{f['log_evidence_error'][()]:.2f}, {f['total_likelihood_evaluations'][()]} likelihood calls, "
              f"sampling time {f['sampling_time'][()]:.0f} s")

print(f"\n{'param':6s} {'truth':>9s} | " + ' | '.join(f"{OG_RUNS[f]:^30s}" for f in OG_RUNS))
for k in ['lat', 'long', 'psi', 's1', 's2']:
    cols = []
    for fac in OG_RUNS:
        q = np.percentile(ogs[fac][k], [5, 50, 95])
        cols.append(f"{q[1]:8.4f} [{q[0]:7.4f},{q[2]:7.4f}] {(q[1] - truth_src[k]) / ogs[fac][k].std():+.2f}")
    print(f"{k:6s} {truth_src[k]:9.4f} | " + ' | '.join(cols))

rng = np.random.default_rng(0)
def og_power(d, t):
    sky = {k: float(d[k]) for k in ENVELOPE_PARAMS}
    return 10 ** (float(d['amp']) - amp_true) * np.stack([np.asarray(power_modulation(sky, t, c)) for c in (0, 1)])
og_idx, P_ogs = {}, {}
for fac, s in ogs.items():
    og_idx[fac] = rng.choice(len(s), 1000, replace=False)
    P_ogs[fac] = np.array([og_power(d, t_fine) for d in s[og_idx[fac]]])            # (ndraw, 2, nt)
P_og = P_ogs[2]
P_wav = 10 ** (amp_ref - amp_true) * modulation_draws(post, r, t_fine, nsamples=1000, seed=0)
print(f'\ndraws: OG {P_og.shape}, wavelet {P_wav.shape}')

In [ ]:
sky = ['lat', 'long', 'psi', 's1', 's2']
rng_ = [tuple(np.percentile(np.r_[ogs[1][k], ogs[2][k]], [0.2, 99.8])) for k in sky]
fig = corner.corner(np.column_stack([ogs[1][k] for k in sky]), labels=sky, range=rng_, color=C_MUTED, bins=30,
                    smooth=1.0, plot_datapoints=False, plot_density=False, levels=(0.5, 0.9), hist_kwargs={'density': True})
corner.corner(np.column_stack([ogs[2][k] for k in sky]), fig=fig, range=rng_, truths=[truth_src[k] for k in sky],
              color=C_OG, truth_color=C_TRUTH, bins=30, smooth=1.0, plot_datapoints=False, plot_density=False,
              levels=(0.5, 0.9), hist_kwargs={'density': True})
fig.legend(handles=[plt.Line2D([], [], color=C_MUTED, label=OG_RUNS[1]), plt.Line2D([], [], color=C_OG, label=OG_RUNS[2])],
           loc='upper right', fontsize=10)
fig.savefig(OUT / 'og_sky_corner.png', dpi=110)

## 4. Spectral parameters: OG vs wavelet

The parameters both models share (`amp` is compared through the galactic power in §5: the wavelet convention
$b_A\equiv0$ shifts it).

In [ ]:
shared = ['alpha', 'fknee', 'fr1', 'fr2', 'A', 'P']
names = [str(n).split(':')[-1] for n in r['psd_names']]
wav = np.column_stack([r['psd_chain'][:, names.index(k)] for k in shared])
og_s = np.column_stack([ogs[2][k] for k in shared])
og1_s = np.column_stack([ogs[1][k] for k in shared])
truths = [truth_src[k] for k in shared]
rng_ = [tuple(np.percentile(np.r_[wav[:, i], og_s[:, i], og1_s[:, i]], [0.2, 99.8])) for i in range(len(shared))]
def kw(ls='-'):    # fresh dicts per call: corner writes the colour into hist_kwargs
    return dict(range=rng_, bins=30, smooth=1.0, plot_datapoints=False, plot_density=False, levels=(0.5, 0.9),
                max_n_ticks=3, labelpad=0.12, hist_kwargs={'density': True, 'linestyle': ls},
                contour_kwargs={'linestyles': ls})
fig = corner.corner(og1_s, labels=shared, color=C_MUTED, **kw('--'))
corner.corner(og_s, fig=fig, color=C_OG, **kw())
corner.corner(wav, fig=fig, color=C_WAV, truths=truths, truth_color=C_TRUTH, **kw())
fig.legend(handles=[plt.Line2D([], [], color=C_MUTED, ls='--', label=f'{OG_RUNS[1]} (og_legacy)'),
                    plt.Line2D([], [], color=C_OG, label=f'{OG_RUNS[2]} (og)'),
                    plt.Line2D([], [], color=C_WAV, label='wavelet RJMCMC')], loc='upper right', fontsize=11)
fig.savefig(OUT / 'corner_spectral_og_vs_wavelet.png', dpi=110, bbox_inches='tight')

print(f"{'':15s} | {'OG, paper dof':^23s} | {'OG, 2 dof':^23s} | {'wavelet':^23s} | wavelet width /")
print(f"{'param':6s} {'truth':>8s} | " + ' | '.join([f"{'median':>8s} {'90% w':>8s} {'z':>5s}"] * 3) + ' | OG    OG(2 dof)')
for i, k in enumerate(shared):
    cols, widths = [], []
    for s_ in (og1_s, og_s, wav):
        q = np.percentile(s_[:, i], [5, 50, 95])
        widths.append(q[2] - q[0])
        cols.append(f'{q[1]:8.3f} {q[2] - q[0]:8.4f} {(q[1] - truths[i]) / s_[:, i].std():+5.2f}')
    print(f'{k:6s} {truths[i]:8.3f} | ' + ' | '.join(cols) + f' | {widths[2] / widths[0]:.2f}  {widths[2] / widths[1]:.2f}')

## 5. The modulation: OG and wavelet posteriors on top of each other

90% (light) and 50% (dark) bands of the galactic power relative to the true amplitude. Dashed: the injected
$M_c^2(t)$. Dotted vertical lines: the boundaries of the 24 OG chunks. Grey: the edge pixels the wavelet likelihood
drops (the wavelet band there is extrapolation). Middle: median / truth − 1 (the band **centres**, comparable now that
both pipelines see the same data). Bottom: 90% width / truth.

In [ ]:
q_og, q_wav, q_og1 = (np.percentile(P, [5, 25, 50, 75, 95], axis=0) for P in (P_og, P_wav, P_ogs[1]))
td = t_fine / DAY
inside = (t_fine >= t_lo) & (t_fine <= t_hi)
summary = {}

fig, axes = plt.subplots(3, 2, figsize=(12, 7.6), sharex=True, gridspec_kw={'height_ratios': [2.2, 1, 1]})
for i, c in enumerate(CH):
    ax, axc, axw = axes[:, i]
    for q, cl, cd, lab in [(q_og, C_OG_L, C_OG, 'OG (2 dof)'), (q_wav, C_WAV_L, C_WAV, 'wavelet RJMCMC')]:
        ax.fill_between(td, q[0, i], q[4, i], color=cl, alpha=0.45, lw=0, label=f'{lab} 90%')
        ax.fill_between(td, q[1, i], q[3, i], color=cd, alpha=0.35, lw=0, label=f'{lab} 50%')
        ax.plot(td, q[2, i], color=cd, lw=1.2)
        axc.fill_between(td, q[0, i] / P_true[i] - 1, q[4, i] / P_true[i] - 1, color=cl, alpha=0.35, lw=0)
        axc.plot(td, q[2, i] / P_true[i] - 1, color=cd, label=lab)
        axw.plot(td, (q[4, i] - q[0, i]) / P_true[i], color=cd, label=lab)
    ax.plot(td, q_og1[0, i], color=C_OG, ls=':', lw=1, label=f'{OG_RUNS[1]} 90%')
    ax.plot(td, q_og1[4, i], color=C_OG, ls=':', lw=1)
    axw.plot(td, (q_og1[4, i] - q_og1[0, i]) / P_true[i], color=C_OG, ls=':', label=OG_RUNS[1])
    ax.plot(td, P_true[i], color=C_TRUTH, ls='--', lw=1.3, label=r'injected $M_c^2(t)$')
    axc.axhline(0, color=C_TRUTH, ls='--', lw=1)
    for a in (ax, axc, axw):
        chunk_lines(a)
        a.axvspan(0, eb * dT / DAY, color=C_MUTED, alpha=0.12, lw=0)
        a.axvspan(td[-1] - eb * dT / DAY, td[-1], color=C_MUTED, alpha=0.12, lw=0)
    ax.set_title(f'channel {c}')
    axw.set_xlabel('time [days]')
    width = lambda q: np.median((q[4, i] - q[0, i])[inside] / P_true[i, inside])
    centre = lambda q: np.median((q[2, i] / P_true[i] - 1)[inside])
    cov = lambda q: np.mean(((q[0, i] <= P_true[i]) & (P_true[i] <= q[4, i]))[inside])
    summary[c] = {name: dict(w=width(q), c=centre(q), cov=cov(q))
                  for name, q in [('OG (paper dof)', q_og1), ('OG (2 dof)', q_og), ('wavelet', q_wav)]}
    print(f'{c}:')
    for name, s in summary[c].items():
        print(f"   {name:17s} median 90% width / P {100 * s['w']:5.1f}%   median offset {100 * s['c']:+5.1f}%   "
              f"truth inside 90%: {100 * s['cov']:3.0f}% of the analysed span")
    print(f"   wavelet / OG width: {summary[c]['wavelet']['w'] / summary[c]['OG (paper dof)']['w']:.2f} (paper dof), "
          f"{summary[c]['wavelet']['w'] / summary[c]['OG (2 dof)']['w']:.2f} (2 dof)")
axes[0, 0].set_ylabel(r'$10^{\log_{10}A-\log_{10}A_{\rm true}}\,P_c(t)$')
axes[1, 0].set_ylabel('median/truth − 1', fontsize=9)
axes[2, 0].set_ylabel('90% width/truth', fontsize=9)
fig.align_ylabels(axes[:, 0])
h, l = axes[0, 0].get_legend_handles_labels()
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.legend(h, l, loc='lower center', ncol=5, fontsize=8)
fig.savefig(OUT / 'modulation_og_vs_wavelet.png', dpi=150, bbox_inches='tight')

### Shape only

The amplitude is common to all times, so part of each band above is the amplitude uncertainty. Dividing every draw
by its own mean over the analysed span removes it and leaves the constraint on the **time dependence** alone.

In [ ]:
def shape(P):
    return P / P[..., inside].mean(axis=-1, keepdims=True)

S_og, S_wav, S_true, S_og1 = shape(P_og), shape(P_wav), shape(P_true), shape(P_ogs[1])
s_og, s_wav, s_og1 = (np.percentile(S, [5, 25, 50, 75, 95], axis=0) for S in (S_og, S_wav, S_og1))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), sharex=True)
for i, (ax, c) in enumerate(zip(axes, CH)):
    for q, cl, cd, lab in [(s_og, C_OG_L, C_OG, 'OG (2 dof)'), (s_wav, C_WAV_L, C_WAV, 'wavelet RJMCMC')]:
        ax.fill_between(td, q[0, i], q[4, i], color=cl, alpha=0.45, lw=0, label=f'{lab} 90%')
        ax.plot(td, q[2, i], color=cd, lw=1.2)
    ax.plot(td, s_og1[0, i], color=C_OG, ls=':', lw=1, label=f'{OG_RUNS[1]} 90%')
    ax.plot(td, s_og1[4, i], color=C_OG, ls=':', lw=1)
    ax.plot(td, S_true[i], color=C_TRUTH, ls='--', lw=1.3, label='injected')
    chunk_lines(ax)
    ax.axvspan(0, eb * dT / DAY, color=C_MUTED, alpha=0.12, lw=0)
    ax.axvspan(td[-1] - eb * dT / DAY, td[-1], color=C_MUTED, alpha=0.12, lw=0)
    ax.set_title(f'channel {c}: $P_c(t)\\,/\\,\\langle P_c\\rangle$')
    ax.set_xlabel('time [days]')
    width = lambda q: np.median((q[4, i] - q[0, i])[inside])
    cov = lambda q: np.mean(((q[0, i] <= S_true[i]) & (S_true[i] <= q[4, i]))[inside])
    print(f'{c} shape: median 90% width  OG (paper dof) {100 * width(s_og1):.1f}%  OG (2 dof) {100 * width(s_og):.1f}%  '
          f'wavelet {100 * width(s_wav):.1f}%  -> wavelet/OG = {width(s_wav) / width(s_og1):.2f}, {width(s_wav) / width(s_og):.2f}'
          f' | truth inside 90%: {100 * cov(s_og1):.0f}%, {100 * cov(s_og):.0f}%, {100 * cov(s_wav):.0f}%')
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / 'shape_og_vs_wavelet.png', dpi=150)

### 5b. The galactic power the data measure

Above, both pipelines use `amp` alone as the galactic normalisation, $10^{\log_{10}A-\log_{10}A_{\rm true}}P_c(t)$. The data
measure the galactic **PSD** in the band, and `amp` trades off against the shape parameters ($\alpha$, $f_{\rm knee}$,
$f_1$, $f_2$), the same in both pipelines (run 1, notebook §7b). Here each draw is multiplied by its galactic spectral
shape relative to the injected one, averaged over the wavelet likelihood's frequency groups with the Fisher weight of
a galactic-power scale, $w_g \propto \nu_g\,[S_{\rm gal}/(S_{\rm gal}+S_n)]^2$ at the injection (per channel), at a common
`amp`: the galactic PSD × $P_c(t)$ where the data constrain it, relative to the truth.

In [ ]:
import jax
import jax.numpy as jnp
from bahamas.wavelet.sampler import _sampled
names_w = [p.name for p in _sampled(post.psd_params)]
theta_inj = np.array([p.value for p in _sampled(post.psd_params)], float)      # amp = amp_ref
Sg0, Sn0, _ = post.spectra(jnp.asarray(theta_inj))
Sg0, Sn0 = np.asarray(Sg0), np.asarray(Sn0)
ok = Sg0 > 1e-3 * Sg0.max()
W = np.stack([np.where(ok, np.asarray(like.nu) * (Sg0 / (Sg0 + Sn0[c])) ** 2, 0.) for c in (0, 1)])
W /= W.sum(1, keepdims=True)
spec_batch = jax.jit(jax.vmap(lambda th: post.spectra(th)[0]))

def shape_ratio(theta):
    # Fisher-weighted galactic spectrum relative to the injection, amp held at amp_ref: (ndraw, 2)
    th = np.array(theta, float); th[:, names_w.index('amp')] = theta_inj[names_w.index('amp')]
    R = np.where(ok, np.asarray(spec_batch(jnp.asarray(th))) / np.where(ok, Sg0, 1.), 0.)
    return R @ W.T

wav_idx = np.random.default_rng(0).choice(len(r['psd_chain']), size=min(1000, len(r['psd_chain'])), replace=False)
R_wav = shape_ratio(r['psd_chain'][wav_idx])
def og_theta(s):
    return np.column_stack([s[k] if k in s.dtype.names else np.zeros(len(s)) for k in names_w])
R_ogs = {fac: shape_ratio(og_theta(ogs[fac][og_idx[fac]])) for fac in ogs}

full = {'OG (paper dof)': P_ogs[1] * R_ogs[1][:, :, None], 'OG (2 dof)': P_ogs[2] * R_ogs[2][:, :, None],
        'wavelet': P_wav * R_wav[:, :, None]}
summary_full = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for i, c in enumerate(CH):
    summary_full[c] = {}
    for name, P in full.items():
        q = np.percentile(P / P_true[None], [5, 50, 95], axis=0)[:, i]
        summary_full[c][name] = dict(w=np.median((q[2] - q[0])[inside]), c=np.median(q[1][inside]) - 1,
                                     cov=np.mean(((q[0] <= 1) & (1 <= q[2]))[inside]))
        if name != 'OG (paper dof)':
            col = C_OG if name.startswith('OG') else C_WAV
            axes[i].fill_between(td, q[0] - 1, q[2] - 1, color=col, alpha=0.25, lw=0)
            axes[i].plot(td, q[1] - 1, color=col, label=name)
    axes[i].axhline(0, color=C_TRUTH, ls='--', lw=1)
    chunk_lines(axes[i])
    axes[i].set_title(f'channel {c}'); axes[i].set_xlabel('time [days]')
    print(f'{c}:')
    for name, s_ in summary_full[c].items():
        print(f"   {name:17s} median 90% width {100 * s_['w']:5.1f}%   median offset {100 * s_['c']:+5.1f}%   "
              f"truth inside 90%: {100 * s_['cov']:3.0f}% of the analysed span")
    print(f"   wavelet / OG (2 dof) width: {summary_full[c]['wavelet']['w'] / summary_full[c]['OG (2 dof)']['w']:.2f}")
axes[0].legend(fontsize=8)
axes[0].set_ylabel('relative to truth')
fig.suptitle(r'Galactic PSD $\times$ $P_c(t)$, Fisher-weighted over frequency')
fig.tight_layout(); fig.savefig(OUT / 'modulation_power_og_vs_wavelet.png', dpi=150)

### Recovered modulation: OG vs wavelet

The recovered $P_c(t)$ of both pipelines against the injected envelope $M_c^2(t)$. As in §5b, each sample's $P_c(t)$ is
put on the injected scale with its amplitude and spectral shape (recovered brightness / injected brightness ×
$M_c^2(t)$), so the dashed curve is the truth. OG: correct (2 dof) likelihood.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={'height_ratios': [2.2, 1]})
for i, c in enumerate(CH):
    ax, axr = axes[:, i]
    for name, cl, cd, lab in [('OG (2 dof)', C_OG_L, C_OG, 'OG chunked (2 dof)'), ('wavelet', C_WAV_L, C_WAV, 'wavelet RJMCMC')]:
        q = np.percentile(full[name][:, i], [5, 25, 50, 75, 95], axis=0)          # on the M_c^2 scale
        ax.fill_between(td, q[0], q[4], color=cl, alpha=0.4, lw=0, label=f'{lab} 90%')
        ax.plot(td, q[2], color=cd, label=f'{lab} median')
        rq = np.percentile(full[name][:, i] / P_true[i], [5, 50, 95], axis=0)
        axr.fill_between(td, rq[0] - 1, rq[2] - 1, color=cl, alpha=0.4, lw=0)
        axr.plot(td, rq[1] - 1, color=cd)
    ax.plot(td, P_true[i], color=C_TRUTH, ls='--', lw=1.3, label=r'injected $M_c^2(t)$')
    axr.axhline(0, color=C_TRUTH, ls='--', lw=1)
    for a in (ax, axr):
        chunk_lines(a)
    ax.set_title(f'channel {c}')
    axr.set_xlabel('time [days]')
axes[0, 0].set_ylabel(r'$P_c(t)$')
axes[1, 0].set_ylabel('recovered / injected − 1')
axes[0, 0].legend(fontsize=8, loc='upper left')
fig.suptitle('Run 2: recovered modulation, OG chunked vs wavelet (same data, no gap)', fontsize=12)
fig.tight_layout()
fig.savefig(OUT / 'modulation_recovered_og_vs_wavelet.png', dpi=150)

## 6. Summary

Compare with the 30-day run 2: [run2_sky_vs_og.ipynb](../../30_days/run2/run2_sky_vs_og.ipynb) and
[reports/WAVELET_INJECTION_RUN2.md](../../../reports/WAVELET_INJECTION_RUN2.md).